# Function Learning Experiment - Large Scale Experiments

**Goal:** Learn a polynomial $f(x) = a_0 + a_1 x + \cdots + a_n x^n$ such that $\sigma(|f(x)|)$ approximates binary labels $y \in \{0, 1\}$.

In [ ]:
include("../../src/NAML.jl")
include("util.jl")

using Oscar
using .NAML

## 1. Global Configuration

Centralized configuration for all experiments. Modify these values to tune the experiments.

In [ ]:
# ============================================================================
# GLOBAL HYPERPARAMETERS - Modify these to configure experiments
# ============================================================================
# NOTE: Default values are based on systematic hyperparameter tuning
# See HYPERPARAMETER_TUNING_RESULTS.md for details

# Basic field configuration
PREC = 20                     # p-adic precision
PRIME = 3                     # prime for p-adic field

# Training configuration
N_EPOCHS = 100                # number of optimization epochs per optimizer
INITIAL_RADIUS = 0            # initial parameter radius (OPTIMAL: 0 for unit ball)
PRINT_EVERY = 100             # print progress every N epochs

# Loss function configuration (OPTIMIZED)
THRESHOLD = 0.7               # sigmoid threshold (OPTIMAL: 0.7, tested: 0.3, 0.5, 0.7)
SCALE = 2.0                   # sigmoid steepness (OPTIMAL: 2.0, tested: 0.5, 1.0, 2.0)
USE_REGULARIZATION = true    # whether to include regularization (OPTIMAL: false)

# Optimizer-specific configuration
MCTS_NUM_SIMULATIONS = 100    # MCTS simulations per step (OPTIMAL: 50-100)
MCTS_EXPLORATION = 1.41       # MCTS exploration constant
GREEDY_DEGREE = 2             # Greedy descent search degree (OPTIMAL: 2)

println("Global configuration loaded (OPTIMIZED DEFAULTS):")
println("  Prime: $PRIME")
println("  Precision: $PREC")
println("  Epochs: $N_EPOCHS")
println("  Initial Radius: $INITIAL_RADIUS (unit ball)")
println("  Threshold: $THRESHOLD, Scale: $SCALE")
println("  Greedy Degree: $GREEDY_DEGREE")
println("  MCTS simulations: $MCTS_NUM_SIMULATIONS")
println("  Use regularization: $USE_REGULARIZATION")

## 2. Task Definitions

Define multiple function learning tasks with different configurations.
Each task has its own dataset and hyperparameters.

In [ ]:
# ============================================================================
# TASK DEFINITIONS - Define multiple learning tasks here
# ============================================================================

# Each task is a named tuple: (name, n_points, poly_degree, min_val, max_val)
# - name: descriptive name for the task
# - n_points: number of training data points
# - poly_degree: degree of polynomial to learn
# - min_val: minimum valuation for random x values
# - max_val: maximum valuation for random x values

tasks = [
    (name="Task1_6pts_deg18", n_points=6, poly_degree=18, min_val=0, max_val=PREC),
    (name="Task2_8pts_deg18", n_points=8, poly_degree=18, min_val=0, max_val=PREC),
    (name="Task3_6pts_deg12", n_points=6, poly_degree=12, min_val=0, max_val=PREC),
]

println("\nConfigured $(length(tasks)) tasks:")
for (i, task) in enumerate(tasks)
    println("  $i. $(task.name):")
    println("     - Points: $(task.n_points)")
    println("     - Degree: $(task.poly_degree)")
    println("     - Valuation range: [$(task.min_val), $(task.max_val)]")
end

## 3. Optimizer Definitions

Define which optimizers to test. Easy to add/remove optimizers.

In [ ]:
# ============================================================================
# OPTIMIZER DEFINITIONS - Configure which optimizers to compare
# ============================================================================

# Define optimizer configurations
# Each optimizer is specified by: (name, init_function, config)

function make_optimizer_configs()
    configs = []
    
    # Greedy descent (baseline)
    push!(configs, (
        name="Greedy_deg$(GREEDY_DEGREE)",
        description="Greedy descent, degree $GREEDY_DEGREE",
        init_fn=(param, loss) -> NAML.greedy_descent_init(param, loss, 1, (false, GREEDY_DEGREE))
    ))
    
    # Greedy descent with adaptive mode
    push!(configs, (
        name="Greedy_adaptive",
        description="Greedy descent with adaptive mode",
        init_fn=(param, loss) -> NAML.greedy_descent_init(param, loss, 1, (true, GREEDY_DEGREE))
    ))
    
    # MCTS
    mcts_config = NAML.MCTSConfig(
        num_simulations=MCTS_NUM_SIMULATIONS,
        exploration_constant=MCTS_EXPLORATION,
        selection_mode=NAML.BestValue,
        degree=2
    )
    push!(configs, (
        name="MCTS",
        description="Monte Carlo Tree Search",
        init_fn=(param, loss) -> NAML.mcts_descent_init(param, loss, mcts_config)
    ))
    
    return configs
end

optimizer_configs = make_optimizer_configs()

println("\nConfigured $(length(optimizer_configs)) optimizers:")
for (i, opt_cfg) in enumerate(optimizer_configs)
    println("  $i. $(opt_cfg.name): $(opt_cfg.description)")
end

## 4. Loss Function Factory

Create loss functions with optional regularization.

In [ ]:
# ============================================================================
# LOSS FUNCTION FACTORY
# ============================================================================

function make_param_diameter_regularization(p::NAML.ValuationPolydisc{S, T})::NAML.Loss where {S, T}
    polydisc_funs = Vector{NAML.PolydiscFunction{S}}()
    prime = NAML.prime(p)
    for i in Base.eachindex(p)
        lam::Function = x -> Float64(prime)^ (- x.radius[i])
        push!(polydisc_funs, NAML.Lambda{S}(lam))
    end
    polydisc_fun = sum(polydisc_funs)
    polydisc_fun = NAML.batch_evaluate_init(polydisc_fun)
    polydisc_fun_batch = function(polydisc_arr::Array{ValuationPolydisc{S, T, M}}) where M 
        return map(polydisc_fun, polydisc_arr)
    end
    return NAML.Loss(polydisc_fun_batch, x -> 0)
end

function create_loss_functions(data, poly_degree, initial_param, use_regularization)
    # Base cross-entropy loss
    base_loss = polynomial_to_crossentropy_loss(data, poly_degree, Float64(THRESHOLD), SCALE)
    
    losses = [
        (name="Unregularized", loss=base_loss)
    ]
    
    # Add regularized version if enabled
    if use_regularization
        regularization = make_param_diameter_regularization(initial_param)
        regularized_loss = base_loss + regularization
        push!(losses, (name="Regularized", loss=regularized_loss))
    end
    
    return losses
end

println("\nLoss function factory ready")
println("  Regularization: $(USE_REGULARIZATION ? "enabled" : "disabled")")

## 5. Run All Experiments

Run all optimizers on all tasks and collect results.

In [ ]:
# ============================================================================
# MAIN EXPERIMENT LOOP
# ============================================================================

# Initialize field once
K = PadicField(PRIME, PREC)

# Storage for all results
# Structure: results[task_idx][loss_idx][optimizer_idx] = result_tuple
all_results = []

# Loop over all tasks
for (task_idx, task) in enumerate(tasks)
    println("\n" * "=" ^ 80)
    println("TASK $(task_idx)/$(length(tasks)): $(task.name)")
    println("=" ^ 80)
    
    # Generate data for this task
    data = generate_polynomial_learning_data(PRIME, PREC, task.n_points, task.min_val, task.max_val)
    println("Generated $(task.n_points) data points")
    
    # Create initial parameter
    initial_param = generate_gauss_point(task.poly_degree + 1, K, INITIAL_RADIUS)
    
    # Create loss functions
    losses = create_loss_functions(data, task.poly_degree, initial_param, USE_REGULARIZATION)
    println("Created $(length(losses)) loss function(s)")
    
    # Storage for this task's results
    task_results = []
    
    # Loop over loss functions
    for (loss_idx, loss_cfg) in enumerate(losses)
        println("\n  Loss: $(loss_cfg.name)")
        
        # Storage for this loss's results
        loss_results = []
        
        # Loop over optimizers
        for (opt_idx, opt_cfg) in enumerate(optimizer_configs)
            println("\n    Optimizer: $(opt_cfg.name)")
            
            # Initialize optimizer
            optim = opt_cfg.init_fn(initial_param, loss_cfg.loss)
            
            # Record initial state
            initial_loss_val = NAML.eval_loss(optim)
            
            # Run optimization
            for epoch in 1:N_EPOCHS
                if epoch % PRINT_EVERY == 0 || epoch == 1
                    current_loss = NAML.eval_loss(optim)
                    println("      Epoch $epoch: loss = $(round(current_loss, digits=6))")
                end
                NAML.step!(optim)
            end
            
            # Collect results
            final_loss_val = NAML.eval_loss(optim)
            final_param = optim.param
            final_center = NAML.center(final_param)
            final_radius = NAML.radius(final_param)
            
            loss_improvement = initial_loss_val - final_loss_val
            improvement_pct = initial_loss_val > 0 ? (loss_improvement / initial_loss_val * 100) : 0.0
            
            # Compute training accuracy
            correct = 0
            for (x, y_true) in data
                f_val = sum(final_center[k] * x^(k-1) for k in 1:length(final_center))
                f_abs = abs(f_val)
                p = sigmoid(f_abs, THRESHOLD, SCALE)
                y_pred = p > 0.5 ? 1.0 : 0.0
                if y_pred == y_true
                    correct += 1
                end
            end
            accuracy = correct / length(data) * 100
            
            # Store result
            result = (
                task_name=task.name,
                loss_name=loss_cfg.name,
                optimizer_name=opt_cfg.name,
                initial_loss=initial_loss_val,
                final_loss=final_loss_val,
                loss_improvement=loss_improvement,
                improvement_pct=improvement_pct,
                accuracy=accuracy,
                final_center=final_center,
                final_radius=final_radius,
                data=data
            )
            push!(loss_results, result)
            
            println("      Final loss: $(round(final_loss_val, digits=6))")
            println("      Improvement: $(round(improvement_pct, digits=2))%")
            println("      Accuracy: $(round(accuracy, digits=2))%")
        end
        
        push!(task_results, loss_results)
    end
    
    push!(all_results, task_results)
end

println("\n" * "=" ^ 80)
println("ALL EXPERIMENTS COMPLETE")
println("=" ^ 80)
println("Total experiments run: $(sum(length(task_results) * length(task_results[1]) for task_results in all_results))")

## 6. Results Summary - Overview Table

High-level comparison of all experiments.

In [ ]:
# ============================================================================
# RESULTS SUMMARY TABLE
# ============================================================================

println("\nRESULTS SUMMARY - ALL EXPERIMENTS")
println("=" ^ 120)
println(rpad("Task", 20) * rpad("Loss", 18) * rpad("Optimizer", 20) * rpad("Initial", 12) * rpad("Final", 12) * rpad("Improv %", 12) * rpad("Accuracy", 10))
println("-" ^ 120)

for task_results in all_results
    for loss_results in task_results
        for result in loss_results
            println(
                rpad(result.task_name, 20) *
                rpad(result.loss_name, 18) *
                rpad(result.optimizer_name, 20) *
                rpad(round(result.initial_loss, digits=2), 12) *
                rpad(round(result.final_loss, digits=2), 12) *
                rpad(round(result.improvement_pct, digits=2), 12) *
                "$(round(result.accuracy, digits=2))%"
            )
        end
    end
end

println("=" ^ 120)

## 7. Results Analysis - Best Performers

Identify the best optimizer for each task.

In [ ]:
# ============================================================================
# BEST PERFORMERS BY TASK
# ============================================================================

println("\nBEST PERFORMERS BY TASK")
println("=" ^ 80)

for (task_idx, task_results) in enumerate(all_results)
    task_name = tasks[task_idx].name
    println("\n$(task_name):")
    
    # Flatten results for this task
    flat_results = vcat([loss_results for loss_results in task_results]...)
    
    # Find best by final loss
    best_loss = argmin([r.final_loss for r in flat_results])
    best_result = flat_results[best_loss]
    println("  Best final loss: $(best_result.optimizer_name) ($(best_result.loss_name))")
    println("    Final loss: $(round(best_result.final_loss, digits=4))")
    println("    Accuracy: $(round(best_result.accuracy, digits=2))%")
    
    # Find best by accuracy
    best_acc = argmax([r.accuracy for r in flat_results])
    best_result_acc = flat_results[best_acc]
    println("  Best accuracy: $(best_result_acc.optimizer_name) ($(best_result_acc.loss_name))")
    println("    Accuracy: $(round(best_result_acc.accuracy, digits=2))%")
    println("    Final loss: $(round(best_result_acc.final_loss, digits=4))")
end

println("\n" * "=" ^ 80)

## 8. Detailed Results - Individual Experiments

Detailed breakdown for each experiment (optional - may be verbose for many experiments).

In [ ]:
# ============================================================================
# DETAILED RESULTS (set SHOW_DETAILED = false to skip)
# ============================================================================

SHOW_DETAILED = true  # Set to true to see full details

if SHOW_DETAILED
    println("\nDETAILED RESULTS")
    println("=" ^ 80)
    
    for task_results in all_results
        for loss_results in task_results
            for result in loss_results
                println("\n" * "=" ^ 80)
                println("Task: $(result.task_name) | Loss: $(result.loss_name) | Optimizer: $(result.optimizer_name)")
                println("=" ^ 80)
                
                println("\nPerformance:")
                println("  Initial loss: $(round(result.initial_loss, digits=6))")
                println("  Final loss: $(round(result.final_loss, digits=6))")
                println("  Improvement: $(round(result.improvement_pct, digits=2))%")
                println("  Training accuracy: $(round(result.accuracy, digits=2))%")
                
                println("\nLearned coefficients (first 5):")
                for (i, coeff) in enumerate(result.final_center[1:min(5, length(result.final_center))])
                    println("  a$(i-1) = $coeff")
                end
                if length(result.final_center) > 5
                    println("  ... ($(length(result.final_center) - 5) more coefficients)")
                end
                
                println("\nParameter radius (first 5):")
                println("  $(result.final_radius[1:min(5, length(result.final_radius))])")
            end
        end
    end
else
    println("\nDetailed results hidden (set SHOW_DETAILED = true to display)")
end

## 9. Export Results (Optional)

Save results to file for further analysis.

In [ ]:
# ============================================================================
# EXPORT RESULTS TO CSV (optional)
# ============================================================================

EXPORT_RESULTS = false  # Set to true to export

if EXPORT_RESULTS
    using Dates
    
    # Create output directory if it doesn't exist
    output_dir = "../../outputs"
    if !isdir(output_dir)
        mkdir(output_dir)
    end
    
    # Generate filename with timestamp
    timestamp = Dates.format(now(), "yyyymmdd_HHMMSS")
    filename = joinpath(output_dir, "function_learning_results_$(timestamp).csv")
    
    # Write CSV header
    open(filename, "w") do io
        write(io, "Task,Loss,Optimizer,Initial_Loss,Final_Loss,Improvement_Pct,Accuracy\n")
        
        # Write results
        for task_results in all_results
            for loss_results in task_results
                for result in loss_results
                    write(io, "$(result.task_name),$(result.loss_name),$(result.optimizer_name),")
                    write(io, "$(result.initial_loss),$(result.final_loss),")
                    write(io, "$(result.improvement_pct),$(result.accuracy)\n")
                end
            end
        end
    end
    
    println("Results exported to: $filename")
else
    println("Results export disabled (set EXPORT_RESULTS = true to enable)")
end

## Summary

This notebook provides a comprehensive framework for large-scale function learning experiments:

**How to Use:**
1. Modify **Section 1** to adjust global hyperparameters
2. Modify **Section 2** to add/remove tasks
3. Modify **Section 3** to add/remove optimizers
4. Run **Section 5** to execute all experiments
5. Review results in **Sections 6-8**

**Large-Scale Experiments:**
- Start with `N_EPOCHS = 100` and `PRINT_EVERY = 50` for quick testing
- Increase epochs gradually: 100 → 300 → 600 → 1000
- For MCTS: 500+ epochs recommended (greedy is faster for short runs)
- Experiment with different task configurations (points, degrees, valuation ranges)
- Use `EXPORT_RESULTS = true` to save data for later analysis